# Pathway 2 (PCA subspace alignment) + regime switching

`pathway2_pca` was the strongest of the five drift-mitigation pathways in `drift_pathways.ipynb`.
This notebook takes that result as the starting point and adds one more idea on top of it:
**regime switching** -- instead of always training on every prior batch, detect which recurring
sensor regime the target batch is actually in (k-means on batch centroids, unlabeled) and train
the PCA-align model on only that regime's source batches. Drift on this dataset is not assumed to
be monotonic: if a later batch's sensors drift back toward an earlier regime, this reuses that
older, better-matched data instead of diluting the fit with the more recent batches in between.

Same forward-chaining protocol as `mlp_final.ipynb` / `drift_pathways.ipynb` (train on
`S_1..S_{T-1}`, validate on `S_T`, report mean and large-drift-fold macro-F1) and the same
balanced-assignment finishing step, so every number here is directly comparable to those
notebooks. Self-contained -- doesn't import from the other notebooks.

Nothing is executed here; run it yourself.

In [1]:
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {DEVICE}"
      + (f"  ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else "  [no GPU]"))

train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
sample_sub = pd.read_csv("data/sample_submission.csv")

FEAT = [f"feat_{i}" for i in range(1, 129)]      # 16 sensors x 8 descriptors, sensor-major
COLS = FEAT + ["concentration"]
N_SENSORS, N_DESC = 16, 8
CLASSES = sorted(train["gas_class"].unique())
K = len(CLASSES)
FOLDS = sorted(train["batch"].unique())[1:]
TEST_QUOTA = 600

_s = StandardScaler().fit(train[FEAT])
_X = _s.transform(train[FEAT])
_c = {b: _X[train["batch"].values == b].mean(0) for b in sorted(train["batch"].unique())}
DRIFT = {b: float(np.linalg.norm(_c[b] - _c[b - 1])) for b in FOLDS}
LARGE_DRIFT = [b for b, s in DRIFT.items() if s >= 5.0]

print(f"large-drift folds: {[int(b) for b in LARGE_DRIFT]}")

device: cuda  (NVIDIA GeForce RTX 5060 Ti)
large-drift folds: [2, 3, 4, 5, 6, 8]


## Shared baseline machinery

Copied verbatim from `drift_pathways.ipynb` / `mlp_final.ipynb` (signed-log prep, plain MLP,
seed-averaged training, macro-F1 helper, balanced-assignment rules) so results here are the same
model, not a reimplementation that might drift from it.

In [2]:
def signed_log(a):
    return np.sign(a) * np.log1p(np.abs(a))


def prep(fit_df, *apply_dfs):
    """signed-log + standardise, fit on the training fold only."""
    sc = StandardScaler().fit(signed_log(fit_df[COLS].values))
    out = [sc.transform(signed_log(d[COLS].values)).astype(np.float32)
           for d in (fit_df,) + apply_dfs]
    return out, sc.scale_[:len(FEAT)].astype(np.float32)


class MLP(nn.Module):
    def __init__(self, n_in, width=256, p_drop=0.3, n_out=K):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, width), nn.BatchNorm1d(width), nn.ReLU(), nn.Dropout(p_drop),
            nn.Linear(width, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(p_drop),
            nn.Linear(128, n_out))

    def forward(self, x):
        return self.net(x)


def train_seeds(X_tr, y_tr, X_ap, feat_scale, n_seeds=5, aug_sigma=0.0,
                epochs=80, bs=256, lr=2e-3, wd=1e-4):
    """Returns per-seed probabilities, shape (n_seeds, len(X_ap), K)."""
    Xt = torch.tensor(X_tr, device=DEVICE)
    Xa = torch.tensor(X_ap, device=DEVICE)
    yt = torch.tensor(y_tr, dtype=torch.long, device=DEVICE)
    scale_t = torch.tensor(feat_scale, device=DEVICE)
    n_feat = len(feat_scale)
    probs = np.zeros((n_seeds, len(X_ap), K), dtype=np.float64)

    for s in range(n_seeds):
        torch.manual_seed(s)
        m = MLP(Xt.shape[1]).to(DEVICE)
        opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=wd)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
        for _ in range(epochs):
            m.train()
            for idx in torch.randperm(len(Xt), device=DEVICE).split(bs):
                if len(idx) < 2:
                    continue
                xb = Xt[idx]
                if aug_sigma > 0 and n_feat == len(FEAT):
                    d = torch.randn(len(idx), N_SENSORS, device=DEVICE) * aug_sigma
                    d = d.repeat_interleave(N_DESC, dim=1)
                    xb = xb.clone()
                    xb[:, :n_feat] += d / scale_t
                loss = F.cross_entropy(m(xb), yt[idx])
                opt.zero_grad(); loss.backward(); opt.step()
            sch.step()
        m.eval()
        with torch.no_grad():
            probs[s] = F.softmax(m(Xa), dim=1).cpu().numpy()
    return probs


def f1_of(P, y):
    return f1_score(y, np.array(CLASSES)[P.argmax(1)], average="macro")


def sinkhorn(P, quota, iters=200, eps=1e-12):
    Q = np.clip(P, eps, None).copy()
    for _ in range(iters):
        Q /= Q.sum(1, keepdims=True)
        Q *= (quota / np.maximum(Q.sum(0), eps))
    return Q


def capped_greedy(P, quota):
    """Most-confident-first under hard per-class caps -> exact counts."""
    remaining = np.array(quota, dtype=int).copy()
    out = np.full(len(P), -1, dtype=int)
    for i in np.argsort(-P.max(1)):
        for c in np.argsort(-P[i]):
            if remaining[c] > 0:
                out[i] = c
                remaining[c] -= 1
                break
    return out


def baseline_fn(tr, ap, seeds=5, aug_sigma=0.0):
    """Plain MLP. What Path 2 and the regime-switch variant are compared against."""
    (Xt, Xa), fscale = prep(tr, ap)
    y = np.searchsorted(CLASSES, tr["gas_class"].values)
    return train_seeds(Xt, y, Xa, fscale, n_seeds=seeds, aug_sigma=aug_sigma).mean(0)

## CV runner

Walks `FOLDS` for any `(train_df, apply_df) -> probs` callable and returns per-fold probabilities
+ labels, so baseline, Path 2, and the regime-switch variant are all scored the same way.

In [3]:
def eval_path(fn, seeds_note=""):
    probs, ys = {}, {}
    t0 = time.time()
    for vb in FOLDS:
        tr = train[train["batch"] < vb]
        va = train[train["batch"] == vb]
        probs[vb] = fn(tr, va)
        ys[vb] = va["gas_class"].values
    print(f"  [{time.time() - t0:.0f}s]" + (f"  {seeds_note}" if seeds_note else ""))
    return probs, ys


def summarize(name, probs, ys):
    per_fold = {vb: f1_of(probs[vb], ys[vb]) for vb in FOLDS}
    return dict(name=name, mean=np.mean(list(per_fold.values())),
                large_drift=np.mean([per_fold[b] for b in LARGE_DRIFT]))


results = []
ALL_PROBS = {}

print("=== baseline: plain MLP, signed-log + standardise ===")
BASE_PROBS, BASE_Y = eval_path(lambda tr, ap: baseline_fn(tr, ap, seeds=5))
ALL_PROBS["baseline"] = BASE_PROBS
results.append(summarize("baseline", BASE_PROBS, BASE_Y))
pd.DataFrame(results)

=== baseline: plain MLP, signed-log + standardise ===
  [81s]


,name,mean,large_drift
0,baseline,0.849122,0.860706


## Path 2 -- subspace alignment (PCA-align)

PCA is fit on the pooled, unlabeled source+target features (no leakage: no labels used), then both
are projected into that shared low-dimensional subspace before the classifier ever sees them.

In [4]:
def prep_pca_align(fit_df, apply_df, n_components=40):
    (Xf, Xa), _ = prep(fit_df, apply_df)
    pca = PCA(n_components=n_components, random_state=0).fit(np.vstack([Xf, Xa]))
    return pca.transform(Xf).astype(np.float32), pca.transform(Xa).astype(np.float32)


def path2_pca_fn(tr, ap, seeds=5, n_components=40):
    Xt, Xa = prep_pca_align(tr, ap, n_components)
    y = np.searchsorted(CLASSES, tr["gas_class"].values)
    fscale = np.ones(n_components, dtype=np.float32)
    return train_seeds(Xt, y, Xa, fscale, n_seeds=seeds).mean(0)


print("=== path 2: PCA subspace alignment ===")
P2_PROBS, P2_Y = eval_path(lambda tr, ap: path2_pca_fn(tr, ap, seeds=5))
ALL_PROBS["path2_pca"] = P2_PROBS
results.append(summarize("path2_pca", P2_PROBS, P2_Y))
pd.DataFrame(results)

=== path 2: PCA subspace alignment ===
  [78s]


,name,mean,large_drift
0,baseline,0.849122,0.860706
1,path2_pca,0.880155,0.898716


## Regime switching on top of Path 2

The 9 train batches are clustered (k-means, k=3) on their standardized feature centroids
(`_c`, unlabeled -- no leakage). For a target batch, its own centroid is scored against those
cluster centers to find its regime, training data is restricted to same-regime source batches,
and Path 2's PCA-align model is fit on just that subset. Cold start (fewer than
`min_regime_batches` same-regime source batches -- e.g. the first time a regime is seen) falls
back to the full pooled fit, same as Path 2 alone.

In [5]:
N_REGIMES = 3
_batch_ids = sorted(train["batch"].unique())
_centroid_mat = np.vstack([_c[b] for b in _batch_ids])
_regime_km = KMeans(n_clusters=N_REGIMES, n_init=10, random_state=0).fit(_centroid_mat)
TRAIN_REGIME = {b: int(l) for b, l in zip(_batch_ids, _regime_km.labels_)}
print(f"regime assignment (train batches): {TRAIN_REGIME}")


def regime_of_df(df):
    """Nearest train-regime for any batch's features (unlabeled, via the k-means centroids above)."""
    centroid = _s.transform(df[FEAT]).mean(0, keepdims=True)
    return int(_regime_km.predict(centroid)[0])


def regime_pca_fn(tr, ap, seeds=5, n_components=40, min_regime_batches=2):
    ap_regime = regime_of_df(ap)
    same_regime = [b for b in tr["batch"].unique() if TRAIN_REGIME[b] == ap_regime]
    tr_regime = tr[tr["batch"].isin(same_regime)]
    if tr_regime["batch"].nunique() < min_regime_batches:
        tr_regime = tr  # cold start: this regime barely seen yet, fall back to the full pool
    return path2_pca_fn(tr_regime, ap, seeds=seeds, n_components=n_components)


print("=== path 2 + regime switching ===")
PR_PROBS, PR_Y = eval_path(lambda tr, ap: regime_pca_fn(tr, ap, seeds=5))
ALL_PROBS["path2_regime"] = PR_PROBS
results.append(summarize("path2_regime", PR_PROBS, PR_Y))
summary_df = pd.DataFrame(results).sort_values("large_drift", ascending=False)
summary_df

regime assignment (train batches): {np.int64(1): 2, np.int64(2): 2, np.int64(3): 0, np.int64(4): 0, np.int64(5): 0, np.int64(6): 1, np.int64(7): 1, np.int64(8): 1, np.int64(9): 1}
=== path 2 + regime switching ===
  [63s]


,name,mean,large_drift
1,path2_pca,0.880155,0.898716
2,path2_regime,0.870733,0.895860
0,baseline,0.849122,0.860706


## Final: refit the best config on all 9 batches, predict batch 10, submit

Picks whichever row of `summary_df` scored highest on the large-drift folds -- baseline, Path 2,
or Path 2 + regime switching -- and reuses that exact function (no retyped logic) to produce the
submission. Raise `seeds` here vs. the CV cells above now that it's one fit instead of nine folds'
worth. Balanced assignment (the test set holds exactly 600 of each class) is applied same as the
other notebooks.

In [6]:
FINAL_SEEDS = 15

candidates = {
    "baseline": lambda tr, ap: baseline_fn(tr, ap, seeds=FINAL_SEEDS),
    "path2_pca": lambda tr, ap: path2_pca_fn(tr, ap, seeds=FINAL_SEEDS),
    "path2_regime": lambda tr, ap: regime_pca_fn(tr, ap, seeds=FINAL_SEEDS),
}

best_name = summary_df.iloc[0]["name"]
print(f"best config by large-drift macro-F1: {best_name}")
P = candidates[best_name](train, test)

quota = np.full(K, TEST_QUOTA)
pred_free = np.array(CLASSES)[P.argmax(1)]
pred = np.array(CLASSES)[capped_greedy(sinkhorn(P, quota), quota)]
print(f"the balanced-assignment constraint changed {(pred != pred_free).mean():.1%} of the 3600 predictions")

sub = pd.DataFrame({"measurement_id": test["measurement_id"], "gas_class": pred})
assert list(sub.columns) == list(sample_sub.columns)
assert len(sub) == len(sample_sub)
assert (sub["measurement_id"].values == sample_sub["measurement_id"].values).all()
assert sub["gas_class"].isin(range(1, 7)).all() and sub["gas_class"].notna().all()
sub.to_csv("data/submission_pathway2_regime.csv", index=False)
print("wrote data/submission_pathway2_regime.csv")

best config by large-drift macro-F1: path2_pca
the balanced-assignment constraint changed 11.9% of the 3600 predictions
wrote data/submission_pathway2_regime.csv
